# 02 · Score a new city with the saved model

**Do not run this for city C before the repo has the tag `v1-AB`.**

The city is a parameter. This notebook **fits nothing**: it loads the model saved by `06_train_evaluate`
and applies it as it is. So the first run on city C is the model exactly as it was for A and B.

For a new city `is_test` is ignored: every listing is unseen, so the error is measured on all `is_trainable` rows.

| Step | Output |
|---|---|
| 1 | Load the saved model (the sha256 shows it is the same file) |
| 2 | Did the data fit the model? Unseen categories, values outside the training range |
| 3 | The three numbers: baseline on the new city, model on the training cities, model on the new city |
| 4 | Why the gap: price level or ranking? Which listings? |
| 5 | Predictions for every listing to `gold_price_predictions` |

In [0]:
dbutils.widgets.text("cities", "munich", "City to score (comma separated for several)")
dbutils.widgets.text("model_version", "v1", "Model version to load")

In [0]:
%run ./00_model_common

# 00 · Model common — shared by training and scoring

Not run on its own: `01_train_evaluate` and `02_score_city` pull it in with `%run ./00_model_common`.

Everything that must be **identical** at training time and at scoring time lives here: the feature lists,
the preprocessing, the baseline and the metric. `02_score_city` never fits anything, it only applies
what `01_train_evaluate` saved. That is what makes the city C run "unchanged".

Input: `workspace.airbnb.gold_listing_features`. Rules we follow from the pipeline:

* target `price_eur`; train and evaluate only where `is_trainable = true`;
* train = `is_test = false`, test = `is_test = true` (the shared split, we never make our own);
* for a new city `is_test` is ignored: every trainable row is unseen;
* never a feature: `listing_id`, `snapshot_date`, `neighbourhood`, `price_missing`, `outlier_class`, `is_trainable`, `is_test`.

## Preprocessing
Plain functions, no fitted state except the list of property types to keep, which is learned on the train rows
and stored with the model. A property type the model has never seen (likely in a new city) becomes `Other`.

## Models
Both predict **log price**; predictions are converted back with `exp`.

* **Gradient boosting** (`HistGradientBoostingRegressor`) with absolute-error loss. The median does not change under
  `log`/`exp`, so the model predicts the median price for a listing, which is exactly what MAE rewards, and no
  correction is needed when converting back. Missing values are handled natively.
* **Ridge regression**: the simple, linear reference. If boosting did not beat it clearly we would keep Ridge.

## Baseline
Median `price_eur` for the same `room_type` and `accommodates`, computed on the **train rows only**.
A combination that does not exist in train falls back to the median of the room type, then to the overall median.

> Participant 4 owns the baseline. This is the same definition, kept here so the model notebook runs on its own.
> If participant 4's function differs, replace `fit_baseline` / `predict_baseline` and re-run: nothing else changes.

## Metric
MAE in EUR (the required metric), plus two numbers that help explain an error:
`bias_eur` = mean of (predicted − actual): negative means the model under-prices on average;
`median_ae_eur` = the error of a typical listing, not pulled up by a few expensive ones.

A small negative bias on the training cities is expected: the model predicts the *median* price of similar
listings, and the median of a right-skewed price is below its mean. A large negative bias in a new city is a
different thing: it means the whole city is more expensive than the model expects.

## Reading and writing tables, saving and loading the model

In [0]:
CITIES = parse_cities(dbutils.widgets.get("cities"))
MODEL_VERSION = dbutils.widgets.get("model_version").strip()

## 1 · Load the saved model and the new city

In [0]:
art = load_artifact(MODEL_VERSION)
FS = art["feature_set"]
overlap = sorted(set(CITIES) & set(art["train_cities"]))
if overlap:
    print(f"NOTE: {overlap} were training cities; their errors below are not 'new city' errors.")

raw = load_features(CITIES)
data = prepare(raw, art["property_types_kept"])
data = data.assign(pred_model=predict_price(art["pipeline"], FS, data),
                   pred_baseline=predict_baseline(art["baseline"], data))
priced = data[data["is_trainable"]].reset_index(drop=True)
print(f"{len(data):,} listings, {len(priced):,} with a usable price (is_trainable)")

model loaded: /Volumes/workspace/airbnb/models/price_model_v1.joblib
sha256: e5f7f5b86262e63915c9f9d5f7d9c1ed2920517d9baa2adf35de8f5e092bf827
trained 2026-11-10 20:33 UTC on ['berlin', 'vienna'] | gbm | core_city_latlon
7,493 listings, 5,076 with a usable price (is_trainable)


## 2 · Did the data fit the model?
The model runs on anything, so "it ran" proves little. These checks show where it was guessing:
categories it never saw, and numeric values outside what it saw in training.

In [0]:
fit_rows = []
for c in [c for c in FEATURE_SETS[FS]["cat"] if c != "property_type"]:      # property_type has its own row below
    seen = set(art["train_categories"][c])
    unseen = ~data[c].isin(seen)
    examples = ", ".join(sorted(data.loc[unseen, c].dropna().astype(str).unique())[:5])
    fit_rows.append({"feature": c, "check": "category not seen in training",
                     "share_of_listings_pct": round(100 * unseen.mean(), 1), "detail": examples})
other = data["property_type"] == "Other"
fit_rows.append({"feature": "property_type", "check": "grouped into 'Other'",
                 "share_of_listings_pct": round(100 * other.mean(), 1),
                 "detail": ", ".join(data.loc[other, "property_type_raw"].value_counts().index[:5].astype(str))})
for c in FEATURE_SETS[FS]["num"]:
    lo, hi = art["train_ranges"][c]
    outside = (data[c] < lo) | (data[c] > hi)
    missing = data[c].isna()
    if outside.mean() >= 0.001 or missing.mean() > 0.05:
        fit_rows.append({"feature": c, "check": "outside training range / missing",
                         "share_of_listings_pct": round(100 * outside.mean(), 1),
                         "detail": f"train {lo:.2f} to {hi:.2f}; here {data[c].min():.2f} to {data[c].max():.2f}; "
                                   f"missing {100 * missing.mean():.1f}%"})
data_fit = pd.DataFrame(fit_rows).sort_values("share_of_listings_pct", ascending=False).reset_index(drop=True)
display(data_fit)

feature,check,share_of_listings_pct,detail
city,category not seen in training,100.0,munich
longitude,outside training range / missing,100.0,train 13.12 to 16.54; here 11.39 to 11.72; missing 0.0%
latitude,outside training range / missing,31.8,train 48.13 to 52.64; here 48.07 to 48.22; missing 0.0%
property_type,grouped into 'Other',2.4,"Private room in townhouse, Entire townhouse, Shared room in rental unit, Tiny home, Private room in tent"
dist_to_centre_km,outside training range / missing,0.5,train 0.13 to 24.62; here 0.02 to 13.56; missing 0.0%
room_type,category not seen in training,0.0,


Baseline fallbacks: how many listings have a room type and guest count that never appeared in train.

In [0]:
known = set(art["baseline"]["by_room_guests"])
fallback = ~pd.Series(list(zip(data["room_type"], data["accommodates"])), index=data.index).isin(known)
print(f"baseline used a fallback for {fallback.sum():,} of {len(data):,} listings ({100 * fallback.mean():.1f}%)")

baseline used a fallback for 10 of 7,493 listings (0.1%)


## 3 · The three numbers, side by side

In [0]:
new_city = error_table(priced, "city", total_label="+".join(CITIES) if len(CITIES) > 1 else None)
trained = pd.DataFrame(art["test_metrics"])
side_by_side = pd.concat([trained.assign(evaluated_on="test (training city)"),
                          new_city.assign(evaluated_on="new city, all priced listings")], ignore_index=True)
display(side_by_side[["city", "evaluated_on", "n_listings", "median_price_eur",
                      "mae_baseline_eur", "mae_model_eur", "improvement_pct", "bias_eur"]])

city,evaluated_on,n_listings,median_price_eur,mae_baseline_eur,mae_model_eur,improvement_pct,bias_eur
berlin,test (training city),1675,128.75,63.8,42.85,32.8,-18.59
vienna,test (training city),1998,124.5,54.85,38.33,30.1,-9.66
berlin+vienna,test (training city),3673,125.57,58.93,40.39,31.5,-13.73
munich,"new city, all priced listings",5076,204.0,163.24,149.61,8.4,-142.97


## 4 · Why the gap?

**Price level or ranking.** If the new city is simply more expensive, the model under-prices almost every
listing by a similar factor (`bias_eur` far below zero). Rescaling the predictions by one number, the ratio of
median actual to median predicted price, removes that part. What is left is the error from ranking listings wrongly.
The rescaling uses the new city's prices, so it is a **diagnosis, not a result**: it is never written to the gold table.

In [0]:
decomp = []
for city, g in priced.groupby("city"):
    factor = g[TARGET].median() / np.median(g["pred_model"])
    mae = np.mean(np.abs(g["pred_model"] - g[TARGET]))
    mae_rescaled = np.mean(np.abs(g["pred_model"] * factor - g[TARGET]))
    decomp.append({
        "city": city,
        "median_price_train_eur": round(art["train_median_price"]["+".join(art["train_cities"])], 2),
        "median_price_here_eur": round(g[TARGET].median(), 2),
        "median_predicted_eur": round(float(np.median(g["pred_model"])), 2),
        "level_factor": round(factor, 3),
        "mae_model_eur": round(mae, 2),
        "mae_after_rescaling_eur": round(mae_rescaled, 2),
        "share_of_mae_from_price_level_pct": round(100 * (1 - mae_rescaled / mae), 1),
        "rank_correlation": round(g["pred_model"].corr(g[TARGET], method="spearman"), 3),
    })
display(pd.DataFrame(decomp))

city,median_price_train_eur,median_price_here_eur,median_predicted_eur,level_factor,mae_model_eur,mae_after_rescaling_eur,share_of_mae_from_price_level_pct,rank_correlation
munich,126.5,204.0,121.4,1.68,149.61,112.83,24.6,0.669


**Which listings.** Error by room type, stay type, size and distance to the centre.

In [0]:
priced = priced.assign(
    stay_type=np.where(priced["is_long_stay"] == 1.0, "long stay (30+ nights)", "short stay"),
    guests=pd.cut(priced["accommodates"], [0, 2, 4, 6, 100], labels=["1-2", "3-4", "5-6", "7+"]).astype(str),
    distance=pd.cut(priced["dist_to_centre_km"], [-0.01, 2, 5, 10, 1e6],
                    labels=["a) 0-2 km", "b) 2-5 km", "c) 5-10 km", "d) 10+ km"]).astype(str),
)
for key in ["room_type", "stay_type", "guests", "distance"]:
    display(error_table(priced, key))

room_type,n_listings,mae_baseline_eur,mae_model_eur,improvement_pct,median_ae_eur,bias_eur,median_price_eur
Entire home/apt,3221,192.05,173.44,9.7,108.67,-166.05,257.0
Hotel room,54,47.71,49.26,-3.2,24.68,-40.72,133.0
Private room,1789,115.43,110.25,4.5,66.12,-104.99,150.0
Shared room,12,77.41,70.77,8.6,53.16,-70.77,104.84


stay_type,n_listings,mae_baseline_eur,mae_model_eur,improvement_pct,median_ae_eur,bias_eur,median_price_eur
long stay (30+ nights),125,49.11,45.37,7.6,18.37,-40.85,74.59
short stay,4951,166.12,152.24,8.4,90.34,-145.55,209.0


guests,n_listings,mae_baseline_eur,mae_model_eur,improvement_pct,median_ae_eur,bias_eur,median_price_eur
1-2,2950,107.64,98.93,8.1,57.41,-92.33,155.0
3-4,1456,192.7,175.5,8.9,129.86,-169.39,275.44
5-6,445,312.69,279.89,10.5,209.35,-273.04,400.0
7+,225,405.99,388.8,4.2,273.4,-378.75,526.2


distance,n_listings,mae_baseline_eur,mae_model_eur,improvement_pct,median_ae_eur,bias_eur,median_price_eur
a) 0-2 km,1340,241.1,217.32,9.9,146.82,-213.24,293.16
b) 2-5 km,2001,153.66,142.0,7.6,83.06,-136.38,200.0
c) 5-10 km,1516,119.25,111.18,6.8,59.6,-101.02,168.68
d) 10+ km,219,78.87,70.75,10.3,45.62,-63.79,125.0


## 5 · Write predictions and metrics
Every listing of the city, also those without a price.

In [0]:
assert np.isfinite(data["pred_model"]).all() and (data["pred_model"] > 0).all()
write_predictions(data, MODEL_VERSION)

metrics = pd.concat(
    [new_city.assign(segment="all")]
    + [error_table(g, key).rename(columns={key: "segment"}).assign(city=city)
       for city, g in priced.groupby("city") for key in ["room_type", "stay_type"]],
    ignore_index=True).assign(evaluated_on="new_city")
write_metrics(metrics, MODEL_VERSION)

workspace.airbnb.gold_price_predictions: wrote 7,493 rows for ['munich'], model_version=v1
workspace.airbnb.gold_model_metrics: wrote 7 rows, model_version=v1


## Summary for the report

In [0]:
ab = trained.iloc[-1]
print(f"Model {MODEL_VERSION} ({art['model_name']}), trained {art['trained_at']} on {art['train_cities']}, applied unchanged\n")
print(f"model on {ab['city']} (test): MAE {ab['mae_model_eur']:.2f} EUR")
for r in new_city.to_dict("records"):
    print(f"baseline on {r['city']}: MAE {r['mae_baseline_eur']:.2f} EUR")
    print(f"model on {r['city']}:    MAE {r['mae_model_eur']:.2f} EUR  (bias {r['bias_eur']:+.2f} EUR, n = {r['n_listings']:,})")

Model v1 (gbm | core_city_latlon), trained 2026-11-10 20:33 UTC on ['berlin', 'vienna'], applied unchanged

model on berlin+vienna (test): MAE 40.39 EUR
baseline on munich: MAE 163.24 EUR
model on munich:    MAE 149.61 EUR  (bias -142.97 EUR, n = 5,076)
